# P3 design-probe: какое правило записи плотной M лучше?

**Без обучения.** Строим плотную per-(user, item) `M` из дневного потока интереса разными правилами
и ранжируем item'ы для предсказания **тестовой** метки (NDCG@10), по **warmth-когортам**, на trade+genre.

Правила (спектр длины памяти):
- `persistent (K=1)` — последнее активное значение (= last);
- `EMA α=0.5`, `EMA α=0.2` — экспоненциальное затухание (короче/длиннее память);
- `uniform MA (K=∞)` — равномерное среднее по всей истории.

**Вопрос:** выигрывает короткая (persistent) или длинная (uniform) память, и **зависит ли это от
тёплости** юзера? Если да — обоснован обучаемый per-user decay `λ_u` (GLA-аналог). Каузальность —
через `join_asof` (строго до дня предсказания). Это де-рискует выбор правила записи ДО написания модели.

In [1]:
import polars as pl, numpy as np, plotly.express as px
from sklearn.metrics import ndcg_score

DS = "/Users/aleksandrpanysev/miniconda3/envs/tgb/lib/python3.13/site-packages/tgb/datasets"
COLS = {"tgbn-trade": ("year", "nation", "trading nation"), "tgbn-genre": ("ts", "user_id", "genre")}
RULES = ["v_persist", "v_ema05", "v_ema02", "v_uniform"]
LBL = {"v_persist": "persistent (K=1)", "v_ema05": "EMA α=0.5",
       "v_ema02": "EMA α=0.2", "v_uniform": "uniform MA (K=∞)"}

def load_norm(name):
    ts, u, it = COLS[name]
    lab = pl.read_csv(f"{DS}/{name.replace('-', '_')}/{name}_node_labels.csv").select(
        [pl.col(ts).alias("ts"), pl.col(u).alias("user"), pl.col(it).alias("item"), pl.col("weight")])
    wsum = lab.group_by(["ts", "user"]).agg(pl.col("weight").sum().alias("wsum"))
    return lab.join(wsum, on=["ts", "user"]).with_columns((pl.col("weight") / pl.col("wsum")).alias("p"))

def probe(name, n=4000, seed=1):
    norm = load_norm(name)
    uts = (norm.select(["user", "ts"]).unique().sort(["user", "ts"])
              .with_columns(pl.int_range(pl.len()).over("user").alias("warmth")))
    items = norm["item"].unique().to_list(); G = len(items)
    imap = pl.DataFrame({"item": items, "ii": list(range(G))})
    recs = (norm.sort(["user", "item", "ts"]).with_columns([
                pl.col("p").alias("v_persist"),
                pl.col("p").cum_sum().over(["user", "item"]).alias("v_uniform"),
                pl.col("p").ewm_mean(alpha=0.5).over(["user", "item"]).alias("v_ema05"),
                pl.col("p").ewm_mean(alpha=0.2).over(["user", "item"]).alias("v_ema02"),
            ]).select(["user", "item", "ts"] + RULES).sort("ts"))
    user_items = norm.select(["user", "item"]).unique()
    tsu = norm["ts"].unique().sort(); t85 = tsu[int(len(tsu) * 0.85)]   # тест-регион
    pairs = uts.filter((pl.col("warmth") >= 1) & (pl.col("ts") >= t85))
    S = pairs.sample(n=min(n, pairs.height), seed=seed).with_row_index("sid")
    warmth_arr = S["warmth"].to_numpy()

    def dense(df, val):
        a = np.zeros((S.height, G)); a[df["sid"].to_numpy(), df["ii"].to_numpy()] = df[val].to_numpy(); return a

    Yt = dense(S.join(norm.select(["user", "ts", "item", "p"]), on=["user", "ts"]).join(imap, on="item"), "p")
    SI = S.select(["sid", "user", "ts"]).join(user_items, on="user").sort("ts")
    pred = (SI.join_asof(recs, on="ts", by=["user", "item"], strategy="backward",
                         allow_exact_matches=False).join(imap, on="item"))
    out = []
    for r in RULES:
        sub = pred.filter(pl.col(r).is_not_null())
        Pk = dense(sub.select(["sid", "ii", r]).rename({r: "v"}), "v")
        keep = np.where((Yt.sum(1) > 0) & (Pk.sum(1) > 0))[0]
        nd = np.array([ndcg_score(Yt[i:i+1], Pk[i:i+1], k=10) for i in keep])
        out.append(pl.DataFrame({"rule": [LBL[r]] * len(keep), "warmth": warmth_arr[keep], "ndcg": nd}))
    return pl.concat(out)

res_trade = probe("tgbn-trade")
print("trade — средний NDCG@10 по правилу (persistent≈0.85 = Table 1 Persistent(L)):")
print(res_trade.group_by("rule").agg(pl.col("ndcg").mean().round(3), pl.len().alias("n")).sort("ndcg", descending=True))

/var/folders/75/6vv7nr6n1cz5sbbl_2m6hfpc0000gn/T/ipykernel_21202/765757176.py:40: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  pred = (SI.join_asof(recs, on="ts", by=["user", "item"], strategy="backward",


trade — средний NDCG@10 по правилу (persistent≈0.85 = Table 1 Persistent(L)):
shape: (4, 3)
┌──────────────────┬───────┬──────┐
│ rule             ┆ ndcg  ┆ n    │
│ ---              ┆ ---   ┆ ---  │
│ str              ┆ f64   ┆ u32  │
╞══════════════════╪═══════╪══════╡
│ persistent (K=1) ┆ 0.789 ┆ 1100 │
│ EMA α=0.5        ┆ 0.784 ┆ 1100 │
│ EMA α=0.2        ┆ 0.764 ┆ 1100 │
│ uniform MA (K=∞) ┆ 0.741 ┆ 1100 │
└──────────────────┴───────┴──────┘


In [2]:
res_genre = probe("tgbn-genre")
print("genre — средний NDCG@10 по правилу:")
print(res_genre.group_by("rule").agg(pl.col("ndcg").mean().round(3), pl.len().alias("n")).sort("ndcg", descending=True))

/var/folders/75/6vv7nr6n1cz5sbbl_2m6hfpc0000gn/T/ipykernel_21202/765757176.py:40: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  pred = (SI.join_asof(recs, on="ts", by=["user", "item"], strategy="backward",


genre — средний NDCG@10 по правилу:
shape: (4, 3)
┌──────────────────┬───────┬──────┐
│ rule             ┆ ndcg  ┆ n    │
│ ---              ┆ ---   ┆ ---  │
│ str              ┆ f64   ┆ u32  │
╞══════════════════╪═══════╪══════╡
│ uniform MA (K=∞) ┆ 0.507 ┆ 4000 │
│ EMA α=0.2        ┆ 0.296 ┆ 4000 │
│ EMA α=0.5        ┆ 0.279 ┆ 4000 │
│ persistent (K=1) ┆ 0.244 ┆ 4000 │
└──────────────────┴───────┴──────┘


In [3]:
def cohort_table(res, nbins=4):
    d = res.with_columns(pl.col("warmth").qcut(nbins, allow_duplicates=True).alias("bin"))
    return (d.group_by(["rule", "bin"]).agg(pl.col("ndcg").mean().round(3).alias("ndcg"),
                                            pl.col("warmth").mean().round(0).alias("aw"),
                                            pl.len().alias("n")).sort(["aw", "rule"]))

ct_trade = cohort_table(res_trade)
print("trade — NDCG@10 по (правило × тёплость):")
print(ct_trade)
fig = px.line(ct_trade.with_columns(pl.col("aw").cast(pl.Int64).cast(pl.Utf8)).to_pandas(),
              x="aw", y="ndcg", color="rule", markers=True,
              title="trade: NDCG@10 правила записи vs тёплость юзера",
              labels={"aw": "тёплость (≈ #дней истории)", "ndcg": "NDCG@10"})
fig.show()

trade — NDCG@10 по (правило × тёплость):
shape: (16, 5)
┌──────────────────┬────────────┬───────┬──────┬─────┐
│ rule             ┆ bin        ┆ ndcg  ┆ aw   ┆ n   │
│ ---              ┆ ---        ┆ ---   ┆ ---  ┆ --- │
│ str              ┆ cat        ┆ f64   ┆ f64  ┆ u32 │
╞══════════════════╪════════════╪═══════╪══════╪═════╡
│ EMA α=0.2        ┆ (-inf, 25] ┆ 0.709 ┆ 21.0 ┆ 396 │
│ EMA α=0.5        ┆ (-inf, 25] ┆ 0.724 ┆ 21.0 ┆ 396 │
│ persistent (K=1) ┆ (-inf, 25] ┆ 0.733 ┆ 21.0 ┆ 396 │
│ uniform MA (K=∞) ┆ (-inf, 25] ┆ 0.7   ┆ 21.0 ┆ 396 │
│ EMA α=0.2        ┆ (25, 26]   ┆ 0.791 ┆ 26.0 ┆ 178 │
│ …                ┆ …          ┆ …     ┆ …    ┆ …   │
│ uniform MA (K=∞) ┆ (26, 28]   ┆ 0.767 ┆ 27.0 ┆ 352 │
│ EMA α=0.2        ┆ (28, inf]  ┆ 0.794 ┆ 29.0 ┆ 174 │
│ EMA α=0.5        ┆ (28, inf]  ┆ 0.823 ┆ 29.0 ┆ 174 │
│ persistent (K=1) ┆ (28, inf]  ┆ 0.819 ┆ 29.0 ┆ 174 │
│ uniform MA (K=∞) ┆ (28, inf]  ┆ 0.763 ┆ 29.0 ┆ 174 │
└──────────────────┴────────────┴───────┴──────┴─────┘


In [4]:
pl.Config.set_tbl_rows(40)
ct_genre = cohort_table(res_genre, nbins=4)
print("genre — NDCG@10 по (правило × тёплость):")
print(ct_genre.sort(["aw", "ndcg"], descending=[False, True]))
fig = px.line(ct_genre.with_columns(pl.col("aw").cast(pl.Int64).cast(pl.Utf8)).to_pandas(),
              x="aw", y="ndcg", color="rule", markers=True,
              title="genre: NDCG@10 правила записи vs тёплость юзера",
              labels={"aw": "тёплость (≈ #дней истории)", "ndcg": "NDCG@10"})
fig.show()

genre — NDCG@10 по (правило × тёплость):
shape: (16, 5)
┌──────────────────┬────────────────┬───────┬───────┬──────┐
│ rule             ┆ bin            ┆ ndcg  ┆ aw    ┆ n    │
│ ---              ┆ ---            ┆ ---   ┆ ---   ┆ ---  │
│ str              ┆ cat            ┆ f64   ┆ f64   ┆ u32  │
╞══════════════════╪════════════════╪═══════╪═══════╪══════╡
│ uniform MA (K=∞) ┆ (-inf, 148.75] ┆ 0.502 ┆ 70.0  ┆ 1000 │
│ EMA α=0.2        ┆ (-inf, 148.75] ┆ 0.318 ┆ 70.0  ┆ 1000 │
│ EMA α=0.5        ┆ (-inf, 148.75] ┆ 0.308 ┆ 70.0  ┆ 1000 │
│ persistent (K=1) ┆ (-inf, 148.75] ┆ 0.281 ┆ 70.0  ┆ 1000 │
│ uniform MA (K=∞) ┆ (148.75, 355]  ┆ 0.48  ┆ 251.0 ┆ 1001 │
│ EMA α=0.2        ┆ (148.75, 355]  ┆ 0.251 ┆ 251.0 ┆ 1001 │
│ EMA α=0.5        ┆ (148.75, 355]  ┆ 0.242 ┆ 251.0 ┆ 1001 │
│ persistent (K=1) ┆ (148.75, 355]  ┆ 0.215 ┆ 251.0 ┆ 1001 │
│ uniform MA (K=∞) ┆ (355, 629.25]  ┆ 0.496 ┆ 491.0 ┆ 999  │
│ EMA α=0.2        ┆ (355, 629.25]  ┆ 0.272 ┆ 491.0 ┆ 999  │
│ EMA α=0.5        ┆ (355, 62

## Выводы design-probe (правило записи M)

**Средний NDCG@10 по правилу (тест-сплит):**

| правило | trade | genre |
|---|---|---|
| persistent (K=1) | **0.789** | 0.244 |
| EMA α=0.5 | 0.784 | 0.279 |
| EMA α=0.2 | 0.764 | 0.296 |
| uniform MA (K=∞) | 0.741 | **0.507** |

**1. Оптимум противоположен между датасетами.** trade → **короткая** память (persistent, λ→0): свежий год
лучший, старые разбавляют. genre → **длинная** память (uniform, λ→1): стабильный долгосрочный вкус
важнее свежести, причём даже мягкий EMA (0.30) сильно хуже равномерного среднего (0.51). →
**одно фиксированное правило записи неверно; нужно ОБУЧАЕМОЕ затухание λ** (может достичь обоих
экстремумов). Подтверждает backlog **H5**.

**2. Внутри genre порядок стабилен по тёплости** (uniform лучший во всех когортах 0.48–0.55,
persistent худший). → главный рычаг — **per-датасет/глобальное** λ; per-**user** λ (GLA-стиль) этим
пробом сильно НЕ подтверждается. *Caveat:* тест-когорты genre почти все тёплые (warmth ≥ 70 в самом
холодном бине) — по-настоящему холодных (warmth 1–10) здесь мало, так что пользу per-user λ для
истинно холодных проб НЕ исключает.

**3. Калибровка.** genre uniform MA **0.507 ≈ Table 1 MovAvg(L) 0.509** → проба достоверна и разрешает
аномалию P2b (0.85): отличие было в (а) тест-сплите и (б) предсказании по ВСЕМ item'ам через
`join_asof` (а не только по сегодня-активным), что корректно штрафует ложные топ-айтемы.

**4. Связь с обзором.** Для scalar exact-index ячейки delta-rule = EMA (нет интерференции ключей —
это и есть entity-space caveat обзора), так что реальный регулятор — **λ**, не delta-vs-additive.

### Следствия для backlog
- **H5/T7:** правило записи M = **обучаемое λ** (диапазон от persistent λ→0 до uniform λ→1);
  начинать с глобального/per-датасет обучаемого скаляра, per-row (GLA) — опционально. persistent
  как дефолт плох на genre; uniform плох на trade.
- **Метрический ориентир:** entity-indexed uniform-M уже даёт genre 0.507 / trade 0.74 в NDCG@10 —
  это «потолок не-обучаемой плотной M»; обучаемая модель (T2/T3) должна его бить, особенно на trade.